# [16.6] SHAP vs Activation Patching - Solutions

## Core question

When do Shapley values and activation patching effects tell the same story, and when does patching overcount interaction credit?

## Learning objectives

By the end, you should be able to:

1. Enumerate complete finite coalition tables.
2. Compute exact Shapley values from weighted marginal contributions.
3. Compute full-minus-ablated patching effects.
4. Verify exact agreement on an additive positive control.
5. Detect and document interaction overcount on an AND negative control.
6. Interpret the CUDA model-organism report without claiming method equivalence.

> Difficulty: 3/5  
> Importance: 4/5

<img src="../../instructions/assets/shap_vs_patching_validation_loop.svg" width="760">

This notebook is the same exercise ladder with the reference implementations filled in. Run all cells to reproduce the local expected outputs and inspect the committed CUDA report.


## Setup

Run this once. The tests are deterministic and small; the final CUDA cell can rerun the model-organism path from the solution module when you want live GPU evidence.

<details><summary>Expected output</summary>

No printed output. Imports should succeed.

</details>


In [1]:
from collections.abc import Callable, Mapping
import itertools
import json
import math
import sys
from pathlib import Path

import torch as t

chapter = "chapter16_shapley_attribution_baselines"
section = "part6_shap_vs_activation_patching"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part6_shap_vs_activation_patching.tests as tests

Coalition = frozenset[int]
MAIN = True


## Exercise 1 - enumerate finite games

Implement `all_coalitions`. Missing even one coalition makes the rest of the attribution comparison meaningless.

<details><summary>Expected output</summary>

```text
All tests in `test_all_coalitions_contract` passed!
```

</details>

<details><summary>Help - what should be included?</summary>

For three players, you should return eight unique coalitions: the empty coalition, three singletons, three pairs, and the full coalition.

</details>

<details><summary>Common bugs</summary>

- Returning only non-empty coalitions.
- Using mutable sets as dictionary keys.
- Missing the full coalition.

</details>

<details><summary>Solution</summary>

Loop over coalition sizes and use `itertools.combinations` to build `frozenset` keys.

</details>


In [2]:
def all_coalitions(num_players: int) -> tuple[Coalition, ...]:
    """Return every player subset for a finite cooperative game."""
    if num_players <= 0:
        raise ValueError("num_players must be positive.")
    coalitions: list[Coalition] = []
    for size in range(num_players + 1):
        coalitions.extend(
            frozenset(group) for group in itertools.combinations(range(num_players), size)
        )
    return tuple(coalitions)


if MAIN:
    tests.test_all_coalitions_contract(all_coalitions)


All tests in `test_all_coalitions_contract` passed!


## Exercise 2 - exact Shapley values

Implement complete-table validation and exact weighted marginal credit.

<details><summary>Expected output</summary>

```text
All tests in `test_exact_shapley_values_additive_toy` passed!
```

</details>

<details><summary>Help - what is the toy oracle?</summary>

For additive weights `[1.0, 2.0, 0.5]`, exact Shapley values must be `[1.0, 2.0, 0.5]`.

</details>

<details><summary>Common bugs</summary>

- Using leave-one-out deltas instead of Shapley weights.
- Silently accepting incomplete tables.
- Returning float32 when exact finite-game arithmetic should use float64.

</details>

<details><summary>Solution</summary>

Normalize keys to `frozenset`, require all `2**n` coalitions, then sum weighted marginal effects with the factorial Shapley weight.

</details>


In [3]:
def normalize_coalition_values(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    num_players: int,
) -> dict[Coalition, float]:
    """Normalize coalition keys and require a complete finite coalition table."""
    values = {frozenset(key): float(value) for key, value in coalition_values.items()}
    expected = set(all_coalitions(num_players))
    missing = expected - set(values)
    if missing:
        raise ValueError(f"coalition value table is missing {len(missing)} coalitions.")
    return values


def exact_shapley_values(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    num_players: int,
) -> t.Tensor:
    """Compute exact Shapley values by summing weighted marginal effects."""
    values = normalize_coalition_values(coalition_values, num_players=num_players)
    shapley = t.zeros(num_players, dtype=t.float64)
    denominator = math.factorial(num_players)
    for player in range(num_players):
        others = [candidate for candidate in range(num_players) if candidate != player]
        for size in range(num_players):
            weight = (
                math.factorial(size)
                * math.factorial(num_players - size - 1)
                / denominator
            )
            for group in itertools.combinations(others, size):
                coalition = frozenset(group)
                shapley[player] += weight * (
                    values[coalition | {player}] - values[coalition]
                )
    return shapley


if MAIN:
    tests.test_exact_shapley_values_additive_toy(exact_shapley_values)


All tests in `test_exact_shapley_values_additive_toy` passed!


## Exercise 3 - full-minus-ablated patching effects

Compute patching effects as `v(N) - v(N without i)`.

<details><summary>Expected output</summary>

```text
All tests in `test_activation_patching_effects_full_minus_ablated` passed!
```

</details>

<details><summary>What you should see</summary>

Additive weights give patching effects `[1.0, 2.0, 0.5]`. A two-feature AND game gives `[1.0, 1.0]`.

</details>

<details><summary>Solution</summary>

Read the full coalition once and subtract the coalition value with each player removed.

</details>


In [4]:
def activation_patching_effects(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    num_players: int,
) -> t.Tensor:
    """Compute full-minus-ablated patching effects from a coalition table."""
    values = normalize_coalition_values(coalition_values, num_players=num_players)
    full = frozenset(range(num_players))
    effects = [values[full] - values[full - {player}] for player in range(num_players)]
    return t.tensor(effects, dtype=t.float64)


if MAIN:
    tests.test_activation_patching_effects_full_minus_ablated(
        activation_patching_effects
    )


All tests in `test_activation_patching_effects_full_minus_ablated` passed!


## Helper games

These two tiny games are the controls for the notebook. The additive game is the positive agreement control; the AND game is the negative overcount control.

<details><summary>Expected output</summary>

No printed output. The helpers should define complete finite coalition tables.

</details>


In [5]:
def _additive_game(weights: t.Tensor) -> dict[Coalition, float]:
    players = range(int(weights.numel()))
    values: dict[Coalition, float] = {}
    for coalition in all_coalitions(int(weights.numel())):
        values[coalition] = float(weights[list(coalition)].sum().item()) if coalition else 0.0
    return values


def _and_game() -> dict[Coalition, float]:
    return {
        frozenset(): 0.0,
        frozenset({0}): 0.0,
        frozenset({1}): 0.0,
        frozenset({0, 1}): 1.0,
    }


def _jsonable_report(report: dict) -> dict:
    result = report.copy()
    for key, value in list(result.items()):
        if hasattr(value, "tolist"):
            result[key] = value.tolist()
    return result


## Exercise 4 - additive agreement report

Compare exact Shapley values and full-minus-ablated patching effects on the additive control.

<details><summary>Expected output</summary>

```text
All tests in `test_shapley_patching_comparison_report_additive` passed!
All tests in `test_additive_agreement_smoke_test` passed!
```

</details>

<details><summary>What you should see</summary>

```text
shapley_values: [1.0, 2.0, 0.5]
patching_effects: [1.0, 2.0, 0.5]
max_abs_error: 0.0
top_feature_agrees: True
```

</details>

<details><summary>Common bugs</summary>

- Checking only the top feature and missing magnitude errors.
- Comparing Shapley values to raw ablated values.
- Treating the positive control as optional.

</details>

<details><summary>Solution</summary>

Compute both vectors, compare their maximum absolute difference, and compare the top feature indices.

</details>


In [6]:
def shapley_patching_comparison_report(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    num_players: int,
    tolerance: float = 1e-9,
) -> dict:
    """Compare exact Shapley values to full-minus-ablated patching effects."""
    shapley = exact_shapley_values(coalition_values, num_players=num_players)
    patching = activation_patching_effects(coalition_values, num_players=num_players)
    max_abs_error = float((shapley - patching).abs().max().item())
    shapley_top = int(shapley.argmax().item())
    patching_top = int(patching.argmax().item())
    return {
        "shapley_values": shapley,
        "patching_effects": patching,
        "max_abs_error": max_abs_error,
        "shapley_top_feature": shapley_top,
        "patching_top_feature": patching_top,
        "top_feature_agrees": shapley_top == patching_top,
        "agrees_with_shapley": max_abs_error <= tolerance,
    }


def additive_agreement_smoke_test() -> dict:
    """Return a JSON-serializable additive agreement report."""
    values = _additive_game(t.tensor([1.0, 2.0, 0.5], dtype=t.float64))
    return _jsonable_report(shapley_patching_comparison_report(values, num_players=3))


if MAIN:
    tests.test_shapley_patching_comparison_report_additive(
        shapley_patching_comparison_report
    )
    tests.test_additive_agreement_smoke_test(additive_agreement_smoke_test)


All tests in `test_shapley_patching_comparison_report_additive` passed!
All tests in `test_additive_agreement_smoke_test` passed!


## Exercise 5 - interaction overcount report

Now run the same comparison on a two-feature AND game.

<details><summary>Expected output</summary>

```text
All tests in `test_interaction_patching_failure_report_and_control` passed!
All tests in `test_interaction_failure_smoke_test` passed!
```

</details>

<details><summary>What you should see</summary>

```text
shapley_values: [0.5, 0.5]
patching_effects: [1.0, 1.0]
shapley_total: 1.0
patching_total: 2.0
overcount: 1.0
```

</details>

<details><summary>Interpreting the result</summary>

Patching asks what happens when this component is removed from the full system. Shapley asks how much marginal credit the component gets across all arrival orders. In interaction-heavy settings, those are different questions.

</details>

<details><summary>Solution</summary>

Reuse your Shapley and patching helpers, then compare `sum(patching)` against `sum(shapley)`.

</details>


In [7]:
def interaction_patching_failure_report(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    num_players: int,
    min_overcount: float = 0.5,
) -> dict:
    """Document patching overcount on an interaction-heavy coalition game."""
    shapley = exact_shapley_values(coalition_values, num_players=num_players)
    patching = activation_patching_effects(coalition_values, num_players=num_players)
    shapley_total = float(shapley.sum().item())
    patching_total = float(patching.sum().item())
    overcount = patching_total - shapley_total
    return {
        "shapley_values": shapley,
        "patching_effects": patching,
        "shapley_total": shapley_total,
        "patching_total": patching_total,
        "overcount": overcount,
        "documents_overcount": overcount >= min_overcount,
    }


def interaction_failure_smoke_test() -> dict:
    """Return a JSON-serializable AND-game overcount report."""
    return _jsonable_report(interaction_patching_failure_report(_and_game(), num_players=2))


if MAIN:
    tests.test_interaction_patching_failure_report_and_control(
        interaction_patching_failure_report
    )
    tests.test_interaction_failure_smoke_test(interaction_failure_smoke_test)


All tests in `test_interaction_patching_failure_report_and_control` passed!
All tests in `test_interaction_failure_smoke_test` passed!


## Exercise 6 - notebook contract

Package both local controls into one JSON-serializable report.

<details><summary>Expected output</summary>

```text
All tests in `test_notebook_contract` passed!
```

</details>

<details><summary>Common bugs</summary>

- Returning tensors instead of lists.
- Including the additive control but omitting the interaction failure.
- Treating this local contract as a substitute for the CUDA report.

</details>

<details><summary>Solution</summary>

Return a dictionary with `additive_agreement` and `interaction_failure`.

</details>


In [8]:
def run_smoke_test(cpu: bool = True) -> dict:
    """Package the additive agreement and interaction-failure reports."""
    _ = cpu
    return {
        "additive_agreement": additive_agreement_smoke_test(),
        "interaction_failure": interaction_failure_smoke_test(),
    }


if MAIN:
    tests.test_notebook_contract(run_smoke_test)


All tests in `test_notebook_contract` passed!


## Exercise 7 - CUDA report interpretation

The committed report trains two finite CUDA model organisms: a linear additive model and a nonlinear interaction model. The cell below audits the committed report and exposes `run_gpu_test` for a live rerun.

<details><summary>Expected output</summary>

```text
All tests in `test_committed_gpu_report_records_agreement_and_disagreement_controls` passed!
{
  "additive_max_abs_error": 2.73e-08,
  "interaction_max_abs_error": 1.10,
  "interaction_abs_overcount": 3.40,
  "peak_vram_gb": 0.06
}
```

</details>

<details><summary>Help - what does this prove?</summary>

It proves the same agreement/disagreement pattern on complete finite model outputs generated by real CUDA training. It does not prove arbitrary transformer activation patching is equivalent to, or worse than, Shapley attribution.

</details>


In [9]:
def load_committed_gpu_report() -> dict:
    """Load the committed CUDA report for interpretation inside the notebook."""
    report = json.loads((section_dir / "verification_report.json").read_text())
    return report["metrics"]["gpu_test"]


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    """Run the full CUDA experiment from the section solution module."""
    from chapter16_shapley_attribution_baselines.exercises.part6_shap_vs_activation_patching import solutions

    return solutions.run_gpu_test(max_vram_gb=max_vram_gb)


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    """Alias used by the repository verification harness."""
    return run_gpu_test(max_vram_gb=max_vram_gb)


if MAIN:
    gpu_report = load_committed_gpu_report()
    tests.test_committed_gpu_report_records_agreement_and_disagreement_controls()
    summary = {
        "additive_max_abs_error": gpu_report["additive_max_abs_error"],
        "interaction_max_abs_error": gpu_report["interaction_max_abs_error"],
        "interaction_abs_overcount": gpu_report["interaction_abs_overcount"],
        "peak_vram_gb": gpu_report["peak_vram_gb"],
    }
    print(json.dumps(summary, indent=2))


All tests in `test_committed_gpu_report_records_agreement_and_disagreement_controls` passed!
{
  "additive_max_abs_error": 2.7318795448039168e-08,
  "interaction_max_abs_error": 1.0999997866650424,
  "interaction_abs_overcount": 3.4000007435679445,
  "peak_vram_gb": 0.06259441375732422
}


## Signature Result

<img src="../../instructions/assets/shap_vs_patching_signature_result.svg" width="780">

The positive control and negative control are both necessary. If the additive model failed, the comparison code would be wrong. If the interaction model did not fail, the notebook would not be teaching the key distinction.

| Setting | Shapley | Patching | Interpretation |
|---|---:|---:|---|
| Additive toy | `[1.0, 2.0, 0.5]` | `[1.0, 2.0, 0.5]` | Agreement |
| AND toy | `[0.5, 0.5]` | `[1.0, 1.0]` | Patching overcounts |
| CUDA additive model | max error `2.73e-8` | same top feature | Positive control passes |
| CUDA interaction model | max disagreement `1.10` | overcount `3.40` | Failure mode preserved |

<details><summary>What this section shows</summary>

- Shapley and full-minus-ablated patching agree on additive finite games.
- Patching can overcount interaction credit even when the top feature agrees.
- The CUDA report preserves both facts on trained finite model organisms.

</details>

## Limitations

<details><summary>What this section does not show</summary>

- It does not rank Shapley above activation patching in general.
- It does not make claims about arbitrary transformer circuits.
- It does not replace clean/corrupt prompt-level patching experiments.

</details>

## Bonus / anomaly hunting

- Try a three-way interaction and measure how overcount scales.
- Add redundant features and check whether the top feature still agrees.
- Compare leave-one-out patching against patching from a baseline other than zero.
